# Laboratorio 13 — Mario aprende a jugar solo
## Versión **estudiante**

**Curso:** Machine Learning (EIN143A25) · Paralelo 701
**Fecha:** Miércoles 18 de noviembre de 2026 · Clase 13 · Lab. Central 114
**Horario del lab:** ~11:50–13:35 (105 min)
**Viene de:** deck «Clase 13 - Aprendizaje por refuerzo»: diapo 3 (el bucle agente–entorno), diapo 4 (política, episodio, retorno), diapo 5 (el futuro depende solo del estado actual), diapo 8 (explorar o explotar), diapo 10 (cuando la tabla no alcanza: una red mira la pantalla), diapo 11 (PPO) y diapo 13 (diseño de la recompensa).

| Parte | Tema | Tiempo sugerido |
|---|---|---|
| **Apertura** | Nadie le enseña a Mario | 5 min |
| **M1** | Conocer el juego | 15 min |
| **M2** | Lo que ve el jugador por dentro | 10 min |
| **M3** | Entrena a tu Mario | 30 min |
| **M4** | El Mario del profe: millones de pasos | 20 min |
| **M5** | Explorar y diseñar los puntos | 10 min |
| **M6** | Reflexión | 5 min |
| — | Margen | 10 min |
| **Entregable** | Respuestas, tabla de M4 y hasta dónde llegó tu Mario | al final |
| *Extra* | *La tabla Q en FrozenLake (opcional, no se evalúa)* | *si te sobra tiempo* |

## Objetivos de aprendizaje

1. Ver con tus propios ojos que un jugador que parte apretando botones al azar aprende solo, partida tras partida, a avanzar por el nivel 1-1.
2. Entender qué mira el jugador (las últimas 4 pantallas) y qué recibe a cambio de cada botón (puntos por avanzar).
3. Entrenar **tu propio Mario** durante 15 minutos y compararlo con el **Mario del profe**, que practicó millones de instantes de juego.
4. Ver por qué conviene probar cosas nuevas y cómo los puntos que le damos deciden qué aprende.

> **Qué necesitas.** Python 3.13 en tu computador, con `pip install -r requirements.txt` (trae el juego, que ya incluye todo lo necesario para correr). Basta la CPU. Si el juego no se puede instalar (por ejemplo, en un Colab con Python más antiguo), el notebook entra en **modo sin juego**: se saltan M1–M3 y en M4 ves los videos ya grabados del Mario del profe. Las preguntas se pueden responder igual.

> **Cómo se trabaja.** Completa las celdas marcadas con `# TODO` (reemplaza los `...`). Justo después de cada una hay un **autochequeo**: una celda que te avisa con un mensaje si algo no cuadra. Si pasa, verás un ✔. Las preguntas se responden en la celda «**Respuesta:**» que sigue a cada una. Las celdas que empiezan con `%%juego` necesitan el juego instalado: en el modo sin juego se saltan solas.

In [ ]:
# --- Configuración -----------------------------------------------------------
import sys, time, random
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PY_313 = sys.version_info >= (3, 13)
URL_REPO = "https://raw.githubusercontent.com/diegoalrv/ml-ein143a25/main/13/"
if IN_COLAB:
    if PY_313:
        %pip install -q "gym-super-mario-bros==9.1.0" "stable-baselines3==2.9.2" "imageio[ffmpeg]" opencv-python-headless
    else:
        print(f"⚠ Este Colab tiene Python {sys.version.split()[0]} y el juego de Mario pide Python 3.13 o más.\n"
              "  El lab se corre en tu computador con Python 3.13. Aquí seguimos en MODO SIN JUEGO.")
if not Path("mario_utils.py").exists():  # las herramientas compartidas del lab (vienen junto al notebook)
    import urllib.request
    urllib.request.urlretrieve(URL_REPO + "mario_utils.py", "mario_utils.py")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from IPython.display import Markdown, display
from IPython.core.magic import register_cell_magic
from mario_utils import (HAY_JUEGO, NIVEL, NOMBRES_BOTONES, PASOS_PROFE, cartel, crear_entorno, crear_vec,
                         descargar_modelos_profe, jugar_partida, miles, mostrar_mp4, mostrar_video,
                         nombre_profe, rotular)

SIN_JUEGO = not HAY_JUEGO
if not SIN_JUEGO:
    import gymnasium as gym
    import torch
    from gym_super_mario_bros.actions import RIGHT_ONLY
    from nes_py.wrappers import JoypadSpace
    from stable_baselines3 import PPO
    from stable_baselines3.common.callbacks import BaseCallback

SEED = 42
RAPIDO = False   # True = 1 minuto de entrenamiento y menos partidas, solo para ensayar el código
T0 = time.time()


def fijar_semillas(semilla=SEED):
    """Deja reproducibles random, NumPy y PyTorch."""
    random.seed(semilla)
    np.random.seed(semilla)
    if not SIN_JUEGO:
        torch.manual_seed(semilla)


fijar_semillas()


@register_cell_magic
def juego(linea, celda):
    """Las celdas con %%juego necesitan el juego: en el modo sin juego se saltan."""
    if SIN_JUEGO:
        print("⏭ MODO SIN JUEGO: esta celda se salta.")
    else:
        get_ipython().run_cell(celda).raise_error()


# Paleta del deck «Clase 13 - Aprendizaje por refuerzo»
AZUL, NARANJA, MARINO, GRIS, GRIS_CLARO = "#2e69b2", "#ba6826", "#1b2740", "#8b929c", "#d9dbdf"
plt.rcParams.update({
    "figure.facecolor": "#f6f6f4", "savefig.facecolor": "#f6f6f4", "axes.facecolor": "#fffefb",
    "axes.edgecolor": "#dcdad3", "axes.labelcolor": MARINO, "text.color": MARINO,
    "xtick.color": "#5d6675", "ytick.color": "#5d6675", "axes.grid": True, "grid.color": "#e3e1da",
    "grid.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlecolor": MARINO, "figure.dpi": 100, "font.size": 10.5,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, MARINO, GRIS]),
})
if SIN_JUEGO:
    print("=" * 78 + "\nMODO SIN JUEGO: NO SE PUDO IMPORTAR gym_super_mario_bros.\n"
          "SE SALTAN M1, M2, M3 Y M5. EN M4 VERÁS LOS VIDEOS Y LA CURVA DEL MARIO DEL PROFE.\n"
          "LAS PREGUNTAS SE PUEDEN RESPONDER IGUAL.\n" + "=" * 78)
else:
    print("✔ Juego listo · Python", sys.version.split()[0], "| PyTorch", torch.__version__,
          "| RAPIDO =", RAPIDO, "| Colab =", IN_COLAB)

---
## Apertura — Nadie le enseña a Mario (5 min)

> **En palabras simples (diapo 3).** Hoy nadie le enseña a Mario a jugar. Nadie le dice «salta aquí». Solo le damos puntos cuando avanza hacia la derecha y se los quitamos cuando se muere. Mario prueba, se equivoca miles de veces y de a poco descubre qué le conviene hacer.

Es el bucle de la diapo 3, con los nombres del juego. En los libros, el jugador se llama **agente**; aquí le diremos **el jugador** o **nuestro Mario**.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.set(xlim=(0, 10), ylim=(-0.2, 3.1))
ax.axis("off")


def caja(x, y, titulo, detalle, fondo, texto="white"):
    ax.add_patch(FancyBboxPatch((x, y), 2.6, 1.0, boxstyle="round,pad=0.04,rounding_size=0.15", fc=fondo, ec="none"))
    ax.text(x + 1.3, y + 0.64, titulo, ha="center", va="center", color=texto, fontsize=12, fontweight="bold")
    ax.text(x + 1.3, y + 0.30, detalle, ha="center", va="center", color=texto, fontsize=9.5)


def flecha(x0, y0, x1, y1, color=MARINO, rad=0.0):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2, connectionstyle=f"arc3,rad={rad}"))


caja(0.4, 1.0, "jugador (Mario)", "mira la pantalla y elige", NARANJA)
caja(7.0, 1.0, "juego", "nivel 1-1", AZUL)
flecha(3.0, 1.75, 7.0, 1.75, rad=-0.25)
ax.text(5.0, 2.55, "aprieta un botón", ha="center", color=MARINO, fontsize=11)
flecha(7.0, 1.25, 3.0, 1.25, rad=-0.25)
ax.text(5.0, 0.2, "pantalla nueva + puntos (más si avanzó, menos si murió)", ha="center", color=MARINO, fontsize=11)
ax.set_title("Una y otra vez, miles de veces: así aprende el jugador")
plt.show()

---
## Parte M1 — Conocer el juego (15 min)

> **En palabras simples (diapos 3 y 4).** El juego le muestra una pantalla a Mario, Mario elige uno de 5 botones y el juego le responde con una pantalla nueva y unos puntos. Una **partida** dura hasta que Mario se muere o llega a la bandera. Cada vuelta del bucle es **un instante de juego** (en los libros, un *paso*).

**Consigna**

1. Mira la primera pantalla y los 5 botones que puede apretar Mario.
2. Mira qué devuelve el juego en cada instante.
3. Completa el ciclo de **una partida al azar** y mira jugar al Mario que aprieta botones sin pensar.

In [ ]:
%%juego
env_crudo = JoypadSpace(gym.make(NIVEL), RIGHT_ONLY)   # el juego tal cual, sin preprocesar, con 5 botones
pantalla, info = env_crudo.reset(seed=SEED)

fig, ax = plt.subplots(figsize=(5, 4.8))
ax.imshow(pantalla)
ax.set_title("Esto es lo que ve el jugador al empezar")
ax.axis("off")
plt.show()
print(f"La pantalla es una imagen de {pantalla.shape[0]} × {pantalla.shape[1]} píxeles a color.")

### Los 5 botones

Le dejamos solo botones hacia la derecha para que aprenda más rápido.

In [ ]:
%%juego
botones = pd.DataFrame({"botones del control": [" + ".join(b) for b in RIGHT_ONLY],
                        "en palabras": NOMBRES_BOTONES})
botones.index.name = "número"
display(botones)

### Lo que devuelve cada instante

Cada vez que el jugador aprieta un botón, `env.step(boton)` devuelve 5 cosas: la **pantalla nueva**, los **puntos** de ese instante, si la partida **terminó**, si se **cortó** por tiempo y la **información del juego**. De la información nos importan sobre todo `x_pos` (qué tan lejos va) y `flag_get` (si tocó la bandera).

In [ ]:
%%juego
pantalla, puntos, terminada, cortada, info = env_crudo.step(1)   # botón 1: derecha
que_es = {"x_pos": "qué tan lejos va (píxeles desde el inicio del nivel)",
          "flag_get": "¿tocó la bandera?",
          "life": "vidas que le quedan",
          "time": "reloj del juego",
          "score": "puntaje del juego",
          "coins": "monedas"}
display(pd.DataFrame({"valor": [info[k] for k in que_es], "qué es": list(que_es.values())}, index=list(que_es)))
print(f"Puntos de este instante: {puntos} · ¿terminó la partida? {terminada}")

### TODO 1 — Una partida al azar

Completa el ciclo: en cada instante, elige un botón al azar (`env_crudo.action_space.sample()`), apriétalo con `env_crudo.step(accion)`, suma los puntos y termina cuando la partida se acabe (`terminada or cortada`).

In [ ]:
%%juego
# TODO 1: completa los ... (una partida con botones al azar)
pantalla, info = env_crudo.reset(seed=SEED)
env_crudo.action_space.seed(SEED)
puntos_total, pasos = 0, 0
terminada = cortada = False
while not (terminada or cortada):
    accion = ...                                                   # un botón al azar
    pantalla, puntos, terminada, cortada, info = ...               # apriétalo
    puntos_total += ...                                            # suma los puntos de este instante
    pasos += 1
x_final = info["x_pos"]
print(f"Partida al azar: {pasos} instantes, {puntos_total:.0f} puntos, llegó a x = {x_final}")

In [ ]:
%%juego
# --- Autochequeo: la partida al azar ---
assert pasos > 0 and x_final > 0, \
    "Revisa que estés usando la acción en `env_crudo.step(accion)` y que el ciclo termine con `terminada or cortada`."
assert isinstance(puntos_total, (int, float, np.floating)), "puntos_total debería ser un número: suma `puntos` en cada vuelta."
print(f"✔ Partida al azar bien: {pasos} instantes y llegó a x = {x_final}")

### Mario al azar, 5 partidas

Ahora usamos `jugar_partida(None)` (de `mario_utils.py`): juega una partida apretando botones al azar y la graba. Para no ver videos eternos, la partida se corta si Mario no avanza en 300 instantes.

In [ ]:
%%juego
azar = [jugar_partida(None, seed=s, etiqueta=f"Mario al azar · partida {s + 1}") for s in range(5)]
X_AZAR = float(np.mean([p["x"] for p in azar]))
display(pd.DataFrame({"distancia (x)": [p["x"] for p in azar], "instantes": [p["pasos"] for p in azar],
                      "¿bandera?": ["sí" if p["bandera"] else "no" for p in azar]},
                     index=pd.Index(range(1, 6), name="partida")))
print(f"En promedio, el Mario al azar llega a x = {X_AZAR:.0f}. La bandera está cerca de x = 3.160.")
mostrar_video(azar[0]["cuadros"])

**Pregunta M1.** ¿Hasta dónde llega Mario apretando botones al azar? ¿Qué lo mata casi siempre?

**Respuesta:** (escribe aquí)

---
## Parte M2 — Lo que ve el jugador por dentro (10 min)

> **En palabras simples (diapos 5 y 10).** Una foto sola no basta: en una foto no se sabe si el Goomba viene hacia Mario o se aleja. Por eso le mostramos **las últimas 4 pantallas juntas**. Además las pasamos a gris y las achicamos: así la red tiene menos números que mirar. Hay demasiadas pantallas posibles para una tabla como la de la diapo 9, así que una **red convolucional**, como la de la Clase 11, mira la pantalla y decide.

`crear_entorno()` hace todo eso: 5 botones, cada instante repite el botón durante 4 cuadros del juego, pasa a gris, achica a 84 × 84 y apila las últimas 4 pantallas.

In [ ]:
%%juego
env = crear_entorno()
obs, info = env.reset(seed=SEED)
for _ in range(8):                      # unos instantes caminando, para que las 4 pantallas sean distintas
    obs, *_ = env.step(1)
obs = np.asarray(obs)

fig, axes = plt.subplots(1, 4, figsize=(11, 3.2))
for i, ax in enumerate(axes):
    ax.imshow(obs[i], cmap="gray", vmin=0, vmax=255)
    ax.set_title(["hace 3 instantes", "hace 2 instantes", "hace 1 instante", "ahora"][i], fontsize=10)
    ax.axis("off")
fig.suptitle("Esto es lo que mira la red: las últimas 4 pantallas", fontweight="bold", color=MARINO)
plt.show()
print(f"240 × 256 × 3 = {miles(240 * 256 * 3)} números  →  4 × 84 × 84 = {miles(obs.size)} números")

In [ ]:
%%juego
# --- Autochequeo: lo que recibe la red ---
assert obs.shape == (4, 84, 84), f"La red debería recibir 4 pantallas de 84 × 84 (forma (4, 84, 84)), no {obs.shape}."
assert obs.dtype == np.uint8 and obs.max() > 0, "Las pantallas deberían ser grises de 0 a 255."
print("✔ La red recibe 4 pantallas en gris de 84 × 84")

**Pregunta M2.** ¿Por qué le damos 4 pantallas y no 1? Piensa en el Goomba.

**Respuesta:** (escribe aquí)

---
## Parte M3 — Entrena a tu Mario (30 min)

> **En palabras simples (diapos 10 y 11).** Usamos un método llamado **PPO**. La idea es la de la diapo 11: *si una jugada terminó bien, hacerla más probable*; si terminó mal, menos. Al principio la red elige casi al azar (explora); a medida que junta experiencia, repite lo que le dio puntos. A la manera de elegir botones la llamaremos **su forma de jugar**.

**Consigna**

1. Ejecuta la celda del cronómetro: entrena **por tiempo, no por pasos** (15 minutos; con `RAPIDO = True`, 1 minuto), así funciona igual en cualquier computador.
2. Completa el **TODO 2**: crea el modelo y lánzalo a entrenar. Mientras entrena, cada ~10.000 instantes verás cómo va tu Mario.
3. Mientras tanto, mira el video del Mario del profe que se proyecta adelante.

In [ ]:
%%juego
MINUTOS_ENTRENAMIENTO = 1 if RAPIDO else 15
N_ENTORNOS = 4      # 4 Marios jugando a la vez (en el mismo proceso): junta experiencia más rápido


class Cronometro(BaseCallback):
    """Detiene el entrenamiento al cumplirse el tiempo. Cada ~10.000 instantes avisa cómo va y graba un clip."""

    def __init__(self, minutos, cada=10_000):
        super().__init__()
        self.minutos, self.cada = minutos, cada
        self.historial, self.clips = [], []

    def _on_training_start(self):
        self.t0, self.proximo = time.time(), self.cada

    def _on_step(self):
        if self.num_timesteps >= self.proximo:
            self.proximo += self.cada
            xs = [e["x_pos"] for e in list(self.model.ep_info_buffer)[-20:]]   # últimas 20 partidas de práctica
            x_prom = float(np.mean(xs)) if xs else 0.0
            self.historial.append((self.num_timesteps, x_prom))
            clip = jugar_partida(self.model, max_pasos=150, deterministico=False,   # como cuando practica
                                 etiqueta=f"Tu Mario · paso {miles(self.num_timesteps)}")
            self.clips.append(clip["cuadros"])
            if len(self.clips) > 12:     # para no llenar la memoria: se queda con 1 de cada 2 clips
                self.clips = self.clips[::2]
            print(f"paso {miles(self.num_timesteps)} · tu Mario llega en promedio a x = {x_prom:.0f} "
                  f"(al azar llegaba a x = {X_AZAR:.0f})", flush=True)
        return time.time() - self.t0 < self.minutos * 60


env_vec = crear_vec(N_ENTORNOS)
cronometro = Cronometro(MINUTOS_ENTRENAMIENTO, cada=2_000 if RAPIDO else 10_000)
print(f"Listo: {N_ENTORNOS} Marios en paralelo y un cronómetro de {MINUTOS_ENTRENAMIENTO} min.")

### TODO 2 — Crea tu Mario y entrénalo

Son los mismos valores que usó el profe:

| valor | qué es |
|---|---|
| `learning_rate=2.5e-4` | qué tan grandes son los ajustes en cada vuelta |
| `n_steps=128` | instantes que juega cada Mario antes de ajustar su forma de jugar |
| `batch_size=256` | de a cuántos instantes revisa la experiencia |
| `n_epochs=4` | cuántas veces repasa la misma experiencia |
| `clip_range=0.1` | cuánto puede cambiar su forma de jugar de una vez (poco: así no olvida lo que ya sabía) |
| `ent_coef=0.01` | un empujoncito para que siga probando botones nuevos |
| `gamma=0.99` | cuánto le importan los puntos que vienen más adelante |

Usa `seed=SEED` y `device="cpu"`. Para entrenar, llama a `modelo.learn(...)` con un número de pasos muy grande (`10_000_000`) y `callback=cronometro`: el cronómetro lo detiene a tiempo.

In [ ]:
%%juego
# TODO 2: crea el modelo con los valores de la tabla y entrénalo con el cronómetro
fijar_semillas()
modelo = PPO("CnnPolicy", env_vec, learning_rate=..., n_steps=..., batch_size=..., n_epochs=...,
             clip_range=..., ent_coef=..., gamma=..., seed=SEED, device="cpu")
t_inicio = time.time()
...                                    # modelo.learn(...) con 10_000_000 pasos y callback=cronometro
minutos_reales = (time.time() - t_inicio) / 60
print(f"Tu Mario practicó {miles(modelo.num_timesteps)} instantes en {minutos_reales:.1f} min "
      f"({modelo.num_timesteps / (minutos_reales * 60):.0f} instantes por segundo).")

Para medir a tu Mario, la celda de abajo lo hace jugar **10 partidas como cuando practica**: elige cada botón con sus probabilidades, igual que en el entrenamiento. Así la comparación con el Mario al azar es pareja: los dos tienen algo de suerte en cada partida.

In [ ]:
%%juego
# --- Autochequeo: ¿tu Mario aprendió? ---
MARGEN = 100   # fijado con la ejecución docente, con holgura
assert modelo.num_timesteps > 0, "Tu Mario no practicó nada: ¿ejecutaste `modelo.learn(...)` con `callback=cronometro`?"
mis_partidas = [jugar_partida(modelo, deterministico=False, seed=s, grabar=False) for s in range(10)]
X_MI_MARIO = float(np.mean([p["x"] for p in mis_partidas]))
X_MAX_MI_MARIO = max(p["x"] for p in mis_partidas)
print(f"10 partidas de tu Mario, como cuando practica: x = {[p['x'] for p in mis_partidas]}\n"
      f"promedio {X_MI_MARIO:.0f} (al azar: {X_AZAR:.0f}) · máximo {X_MAX_MI_MARIO}")
if RAPIDO:
    print("⚠ RAPIDO = True: con 1 minuto de práctica no se revisa si aprendió (cambia a RAPIDO = False).")
else:
    assert X_MI_MARIO > X_AZAR + MARGEN, \
        "Tu Mario todavía no aprende. Revisa la tasa de aprendizaje o entrena unos minutos más."
    print(f"✔ Tu Mario llega más lejos que el Mario al azar: x = {X_MI_MARIO:.0f} contra {X_AZAR:.0f}")

### Cómo fue aprendiendo tu Mario

La curva naranja es la distancia promedio de sus últimas partidas de práctica; la línea gris punteada, la del Mario al azar.

In [ ]:
%%juego
pasos_h, x_h = zip(*cronometro.historial) if cronometro.historial else ([0], [0])
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(pasos_h, x_h, color=NARANJA, lw=2.2, marker="o", ms=4, label="tu Mario")
ax.axhline(X_AZAR, color=GRIS, ls="--", lw=1.6, label="Mario al azar")
ax.set(xlabel="instantes de práctica", ylabel="distancia promedio (x)", title="Tu Mario, minuto a minuto")
ax.xaxis.set_major_formatter(lambda v, _: miles(v))
ax.legend()
plt.show()

Denle play y miren cómo cambió su Mario en 15 minutos.

In [ ]:
%%juego
animacion = [c for clip in cronometro.clips for c in clip]
mostrar_video(animacion, max_seg=120) if animacion else print("Todavía no hay clips: entrena al menos 10.000 instantes.")

In [ ]:
%%juego
modelo.save("mi_mario.zip")
print(f"Guardado mi_mario.zip · tu Mario llegó como máximo a x = {X_MAX_MI_MARIO} (anótalo para el entregable)")

**Pregunta M3.** ¿Tu Mario mejoró? ¿Qué aprendió primero: a caminar hacia la derecha, a saltar el Goomba o a saltar los tubos?

**Respuesta:** (escribe aquí)

---
## Parte M4 — El Mario del profe: millones de pasos (20 min)

> **En palabras simples (diapo 8).** El Mario del profe hizo lo mismo que el tuyo, pero durante millones de pasos. Al principio **explora**: prueba botones sin saber qué pasa. Con la práctica, **explota** lo que aprendió: repite lo que le dio puntos.

Primero bajamos sus modelos de la Release `lab13-mario` del repositorio del curso (unos 50 MB; si ya están en `modelos_mario/`, no se vuelven a bajar).

In [ ]:
CARPETA = descargar_modelos_profe(solo_videos=SIN_JUEGO)

### El video del lab

Con cada modelo del profe, una partida determinista (siempre aprieta el botón que cree mejor), todas juntas en un solo video.

In [ ]:
if SIN_JUEGO:
    display(mostrar_mp4(CARPETA / "mario_progreso.mp4", ancho=640))
else:
    profe, video = [], []
    for p in PASOS_PROFE:
        m = PPO.load(CARPETA / f"{nombre_profe(p)}.zip", device="cpu")
        partida = jugar_partida(m, etiqueta=f"Mario del profe · {miles(p)} pasos")
        partida["pasos_practica"] = p
        profe.append(partida)
        video += cartel(f"Después de {miles(p)}\npasos de práctica") + partida["cuadros"]
    display(mostrar_video(video, max_seg=180, ancho=640))

### ¿En qué parte del camino va tu Mario?

La curva azul sale de `progreso_profe.csv`: cada 50.000 pasos, el profe midió la distancia media de 5 partidas **deterministas** de su Mario (siempre el botón que cree mejor, como en el video). Para que la comparación sea pareja, el punto naranja también es tu Mario jugando así. Ojo: en M3 tu Mario elegía con sus probabilidades y por eso allá llegaba más lejos. Esa diferencia la miramos en M5.

In [ ]:
progreso = pd.read_csv(CARPETA / "progreso_profe.csv")
curva = progreso[progreso["pasos"] > 0]


def rotulo_pasos(v, _=None):
    return f"{v / 1e6:g} millón" if v == 1e6 else f"{v / 1e6:g} millones" if v >= 1e6 else f"{v / 1e3:g} mil"


fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(curva["pasos"], curva["x_media"], color=AZUL, lw=2.2, label="Mario del profe")
ax.set_xscale("log")
ax.set_xticks([1e4, 5e4, 1e5, 3e5, 1e6, 2e6] + ([5e6] if curva["pasos"].max() > 2e6 else []))
ax.xaxis.set_major_formatter(rotulo_pasos)
ax.xaxis.set_minor_formatter(lambda v, _: "")
ax.axhline(3160, color=MARINO, ls=":", lw=1.2)
ax.text(curva["pasos"].min(), 3160, " bandera", va="bottom", color=MARINO, fontsize=9)
if "X_AZAR" in globals():
    ax.axhline(X_AZAR, color=GRIS, ls="--", lw=1.6, label="Mario al azar")
if "modelo" in globals():   # tu Mario de M3, jugando determinista como el del profe
    X_MI_MARIO_DET = np.mean([jugar_partida(modelo, espera_inicial=30, seed=s, grabar=False)["x"] for s in range(5)])
    ax.scatter([modelo.num_timesteps], [X_MI_MARIO_DET], color=NARANJA, s=90, zorder=5, label="tu Mario")
    print(f"Tu Mario, jugando determinista: x media = {X_MI_MARIO_DET:.0f} después de {miles(modelo.num_timesteps)} pasos")
ax.set(xlabel="pasos de práctica (escala logarítmica)", ylabel="distancia media (x)",
       title="¿En qué parte del camino va tu Mario?", ylim=(0, 3400))
ax.legend(loc="center left")
plt.show()

In [ ]:
if SIN_JUEGO:
    tabla = progreso[progreso["pasos"].isin(PASOS_PROFE)].assign(
        bandera=lambda d: np.where(d["prop_bandera"] > 0, "sí", "no"))[["pasos", "x_max", "bandera"]]
    tabla.columns = ["pasos de práctica", "distancia (x)", "¿tocó la bandera?"]
    tabla.index = [nombre_profe(p) for p in tabla["pasos de práctica"]]
    for p in PASOS_PROFE:
        display(Markdown(f"**{nombre_profe(p)}**"), mostrar_mp4(CARPETA / f"{nombre_profe(p)}.mp4", ancho=384))
else:
    tabla = pd.DataFrame({"pasos de práctica": [p["pasos_practica"] for p in profe],
                          "distancia (x)": [p["x"] for p in profe],
                          "¿tocó la bandera?": ["sí" if p["bandera"] else "no" for p in profe]},
                         index=[nombre_profe(p["pasos_practica"]) for p in profe])
tabla.index.name = "modelo"
display(tabla.style.format({"pasos de práctica": miles}))

### TODO 3 — ¿Qué aprendió en cada etapa?

Mirando el video, completa la tabla con tus palabras (por ejemplo: «se queda quieto», «camina y choca con el primer Goomba», «salta el Goomba pero se traba en el tubo»…).

| pasos de práctica | qué aprendió a hacer (en tus palabras) |
|---|---|
| 0 | |
| 50.000 | |
| 150.000 | |
| 300.000 | |
| 600.000 | |
| 1.000.000 | |
| 2.000.000 | |

**Pregunta M4.** ¿En qué momento Mario aprende a saltar el primer Goomba? ¿Mejora de forma pareja o a saltos? ¿Por qué crees que necesita tantos intentos?

**Respuesta:** (escribe aquí)

---
## Parte M5 — Explorar y diseñar los puntos (10 min)

> **En palabras simples (diapos 8 y 13).** Si Mario siempre aprieta el botón que cree mejor, juega igual todas las veces. Si a veces prueba otro, puede descubrir algo nuevo… o morirse. Y ojo con los puntos: Mario aprende a hacer lo que le da puntos, no lo que nosotros queríamos.

Tomamos el mejor Mario del profe y lo hacemos jugar 3 partidas **deterministas** (siempre el botón que cree mejor) y 3 **no deterministas** (elige al azar, pero con más probabilidad los botones que cree mejores).

In [ ]:
%%juego
mejor = max(profe, key=lambda p: (p["bandera"], p["x"], -p["pasos_practica"]))
m_mejor = PPO.load(CARPETA / f"{nombre_profe(mejor['pasos_practica'])}.zip", device="cpu")
filas, no_det = [], []
for s in range(3):
    p = jugar_partida(m_mejor, deterministico=True, seed=s, grabar=False)
    filas.append(("determinista", s + 1, p["x"], "sí" if p["bandera"] else "no"))
for s in range(3):
    p = jugar_partida(m_mejor, deterministico=False, seed=s, etiqueta=f"No determinista · partida {s + 1}")
    no_det.append(p)
    filas.append(("no determinista", s + 1, p["x"], "sí" if p["bandera"] else "no"))
print(f"Mejor Mario del profe: {nombre_profe(mejor['pasos_practica'])}")
display(pd.DataFrame(filas, columns=["forma de elegir", "partida", "distancia (x)", "¿bandera?"]))
mostrar_video(no_det[0]["cuadros"])

### ¿De qué están hechos los puntos?

En cada instante, el juego arma los puntos sumando varias piezas: avanzar, el reloj, morir, el puntaje, las monedas, crecer y terminar el nivel. Miremos algunos instantes de dos partidas: la del mejor Mario del profe, que llega a la bandera, y la del Mario de 0 pasos, que choca con el primer Goomba.

In [ ]:
%%juego
def piezas_de_los_puntos(m):
    """Juega una partida determinista y anota, en cada instante, de qué piezas están hechos los puntos."""
    env = crear_entorno()
    obs, info = env.reset(seed=SEED)
    filas, terminada, cortada, t = [], False, False, 0
    while not (terminada or cortada) and t < 2000:
        accion, _ = m.predict(np.asarray(obs), deterministic=True)
        obs, puntos, terminada, cortada, info = env.step(int(accion))
        t += 1
        filas.append({"instante": t, "x": info["x_pos"], **info["reward_components"]})
    return pd.DataFrame(filas).set_index("instante").rename(columns={
        "progress": "avanzar", "time": "reloj", "score": "puntaje", "coins": "monedas",
        "powerup": "crecer", "completion": "terminar nivel", "death": "morir"})


ganador = piezas_de_los_puntos(m_mejor)
otras = ganador[(ganador.drop(columns=["x", "avanzar"]) != 0).any(axis=1)]
tabla = pd.concat([ganador.head(3), otras.head(6), ganador.tail(1)])
display(Markdown("**El mejor Mario del profe** (llega a la bandera):"), tabla[~tabla.index.duplicated()])
novato = piezas_de_los_puntos(PPO.load(CARPETA / f"{nombre_profe(0)}.zip", device="cpu"))
display(Markdown("**El Mario de 0 pasos** (sus últimos instantes, cuando choca con el Goomba):"), novato.tail(3))

**Pregunta M5.** Si le diéramos puntos solo por juntar monedas, ¿qué crees que aprendería a hacer Mario? ¿Y si lo castigáramos solo por morirse?

**Respuesta:** (escribe aquí)

---
## Parte M6 — Reflexión (5 min)

**Pregunta M6.1.** ¿En qué se diferencia cómo aprendió Mario de cómo aprendió la CNN de la Clase 11 a reconocer ropa?

**Respuesta:** (escribe aquí)

**Pregunta M6.2.** ¿Qué parte de este lab es una red neuronal y qué parte es aprendizaje por refuerzo?

**Respuesta:** (escribe aquí)

**Pregunta M6.3.** ¿Dónde más se podría usar esta idea?

**Respuesta:** (escribe aquí)

---
## Entregable

Sube a AULA-USM este notebook (`Lab13_Mario_estudiante.ipynb`) **ejecutado**, con:

1. Las respuestas de M1 a M6.
2. La tabla de M4 completada.
3. **Hasta dónde llegó tu Mario** (el `x` máximo que imprime la celda que guarda `mi_mario.zip`): con eso armamos la tabla del curso para ver quién llegó más lejos.

In [ ]:
%%juego
print(f"Hasta dónde llegó tu Mario: x máximo = {X_MAX_MI_MARIO} (promedio {X_MI_MARIO:.0f}, "
      f"después de {miles(modelo.num_timesteps)} instantes de práctica)")
print(f"Tiempo total del notebook: {(time.time() - T0) / 60:.1f} min")

---
## Extra opcional — La tabla Q en FrozenLake

*No se evalúa.* Si terminaste antes, abre `Lab13_FrozenLake_CartPole_estudiante.ipynb` (en esta misma carpeta). Ahí se ve, celda por celda, la **tabla Q** de la diapo 9: cuando el mundo es chico (un lago de 4 × 4 casillas), el jugador puede anotar en una tabla qué tan bueno es cada botón en cada casilla. En Mario hay demasiadas pantallas posibles para una tabla, y por eso usamos una red.